# Machine Learning with Spark MLlib

**MLlib** is Spark's Machine Learning library. Its `pyspark.ml` API works directly on DataFrames and, like the rest of Spark, scales from a laptop to a cluster without changing the code.

In this notebook we train a model to predict **customer churn**, that is, which customers of a subscription service are going to cancel. The dataset is in `data/input/customers.csv`.

Main steps:

1. Loading and exploring the data
2. Preparing the features (encoding categorical variables, `VectorAssembler`, scaling)
3. Building a `Pipeline` and training a logistic regression
4. Evaluating the model on a test set
5. Comparing it with a Random Forest
6. Tuning the hyperparameters with cross-validation
7. Saving and loading the model

## 1. Setup

In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

In [ ]:
BASE_FILEPATH = Path("/home/jovyan/data")
INPUT_FILEPATH = BASE_FILEPATH / "input"
OUTPUT_FILEPATH = BASE_FILEPATH / "output"

SEED = 42  # fixed seed, so that splits and models are reproducible

In [ ]:
appName = "MLlibChurn"

spark = SparkSession.builder \
    .appName(appName) \
    .getOrCreate()

spark_ui_port = spark.sparkContext.uiWebUrl.rsplit(":", 1)[-1]
print(f"Spark UI: http://localhost:{spark_ui_port}")

## 2. Loading and exploring the data

Each row is a customer:

- **`age`**: age of the customer
- **`plan`**: subscription plan (`basic`, `standard`, `premium`)
- **`tenure_months`**: how many months the customer has been subscribed
- **`monthly_spend`**: average monthly spend
- **`support_tickets`**: support requests opened in the last period
- **`churned`**: the **label** to predict: `1` if the customer cancelled the subscription, `0` otherwise

This time we use `inferSchema` for brevity: all the columns are simple numbers or strings.

In [ ]:
customers_path = str(INPUT_FILEPATH / "customers.csv")

df = spark.read \
    .option(key="header", value=True) \
    .option(key="inferSchema", value=True) \
    .csv(path=customers_path)

df.printSchema()
df.show(n=5)

Before training a model it's worth checking how the label is distributed. If one class were very rare (e.g. 2% of churned customers), accuracy alone would be a misleading metric.

In [ ]:
df.groupBy("churned").count().show()

A first look at the relationship between features and label: the churn rate by plan and by number of support tickets.

In [ ]:
df.groupBy("plan") \
    .agg(
        F.count("*").alias("customers"),
        F.round(F.avg("churned"), 3).alias("churn_rate"),
    ) \
    .orderBy("plan") \
    .show()

df.groupBy("support_tickets") \
    .agg(
        F.count("*").alias("customers"),
        F.round(F.avg("churned"), 3).alias("churn_rate"),
    ) \
    .orderBy("support_tickets") \
    .show()

## 3. Train / test split

The model is trained on one part of the data (**training set**, 80%) and evaluated on the rest (**test set**, 20%), which it has never seen. This is how we estimate how it will behave on new customers.

In [ ]:
df_train, df_test = df.randomSplit(weights=[0.8, 0.2], seed=SEED)

print(f"Training set: {df_train.count()} rows")
print(f"Test set: {df_test.count()} rows")

## 4. Preparing the features

MLlib models expect all features in a **single column of type vector**. The transformations needed are:

- **`StringIndexer`**: converts a categorical column (`plan`) into a numeric index (`0.0`, `1.0`, `2.0`; the most frequent value gets `0.0`)
- **`OneHotEncoder`**: converts the index into a binary vector, so the model doesn't treat the plans as ordered numbers
- **`VectorAssembler`**: combines all the features into a single vector column
- **`StandardScaler`**: scales the features to unit standard deviation, so that features with large values (e.g. `monthly_spend`) don't dominate those with small values (e.g. `support_tickets`)

In [ ]:
numeric_cols = ["age", "tenure_months", "monthly_spend", "support_tickets"]

plan_indexer = StringIndexer(inputCol="plan", outputCol="plan_index")
plan_encoder = OneHotEncoder(inputCols=["plan_index"], outputCols=["plan_vec"])
assembler = VectorAssembler(inputCols=numeric_cols + ["plan_vec"], outputCol="features_raw")
scaler = StandardScaler(inputCol="features_raw", outputCol="features")

## 5. Pipeline and logistic regression

A **`Pipeline`** chains the preparation steps and the model into a single object. Calling `fit()` on the training set trains every step in order and returns a **`PipelineModel`**, which can be applied to new data with `transform()`.

Using a pipeline guarantees that the test set (and later any new data) goes through **exactly the same transformations** as the training set, with the parameters learned on the training set only (e.g. the mean and standard deviation of the scaler).

In [ ]:
lr = LogisticRegression(featuresCol="features", labelCol="churned")

lr_pipeline = Pipeline(stages=[plan_indexer, plan_encoder, assembler, scaler, lr])

lr_model = lr_pipeline.fit(df_train)

The model adds three columns to the DataFrame:

- **`rawPrediction`**: the raw score of each class
- **`probability`**: the probability of each class (`[P(churned=0), P(churned=1)]`)
- **`prediction`**: the predicted class (`1.0` if the probability of churn is above 0.5)

In [ ]:
lr_predictions = lr_model.transform(df_test)

lr_predictions \
    .select("customer_id", "plan", "tenure_months", "support_tickets", "probability", "prediction", "churned") \
    .show(n=10, truncate=False)

## 6. Evaluating the model

- **Accuracy**: the fraction of correct predictions
- **AUC** (Area Under the ROC Curve): how well the model ranks churned customers above the others, regardless of the 0.5 threshold. It goes from 0.5 (random) to 1.0 (perfect).

In [ ]:
auc_evaluator = BinaryClassificationEvaluator(labelCol="churned", metricName="areaUnderROC")
accuracy_evaluator = MulticlassClassificationEvaluator(labelCol="churned", metricName="accuracy")

print(f"Logistic regression - AUC: {auc_evaluator.evaluate(lr_predictions):.3f}")
print(f"Logistic regression - Accuracy: {accuracy_evaluator.evaluate(lr_predictions):.3f}")

The **confusion matrix** shows where the model makes mistakes: rows are the true classes, columns the predicted classes.

In [ ]:
lr_predictions \
    .groupBy("churned") \
    .pivot("prediction") \
    .count() \
    .orderBy("churned") \
    .show()

### Interpreting the coefficients

In a logistic regression each feature has a coefficient: a positive value increases the probability of churn, a negative one decreases it. Since the features are scaled, the coefficients can be compared with each other.

The trained steps are available in `lr_model.stages`. The `OneHotEncoder` drops the last category by default (it would be redundant), so the plan has one fewer column than the number of plans.

In [ ]:
plan_labels = lr_model.stages[0].labelsArray[0]
feature_names = numeric_cols + [f"plan={label}" for label in plan_labels[:-1]]

coefficients = lr_model.stages[-1].coefficients.toArray()

for name, coef in sorted(zip(feature_names, coefficients), key=lambda x: -abs(x[1])):
    print(f"{name:<18} {coef:+.3f}")

## 7. Comparing with a Random Forest

Thanks to the pipeline, trying another model only means replacing the last step. A **Random Forest** combines many decision trees and can capture non-linear relationships between features.

In [ ]:
rf = RandomForestClassifier(featuresCol="features", labelCol="churned", numTrees=50, seed=SEED)

rf_pipeline = Pipeline(stages=[plan_indexer, plan_encoder, assembler, scaler, rf])
rf_model = rf_pipeline.fit(df_train)
rf_predictions = rf_model.transform(df_test)

print(f"Random forest - AUC: {auc_evaluator.evaluate(rf_predictions):.3f}")
print(f"Random forest - Accuracy: {accuracy_evaluator.evaluate(rf_predictions):.3f}")

A Random Forest has no coefficients, but it estimates the **importance** of each feature (the values sum to 1):

In [ ]:
importances = rf_model.stages[-1].featureImportances.toArray()

for name, importance in sorted(zip(feature_names, importances), key=lambda x: -x[1]):
    print(f"{name:<18} {importance:.3f}")

## 8. Hyperparameter tuning with cross-validation

**Hyperparameters** are model settings that aren't learned from the data, such as the regularization strength `regParam` of the logistic regression. To choose them we use **k-fold cross-validation**: the training set is split into *k* parts (folds), the model is trained *k* times on *k-1* parts and evaluated on the remaining one, and the best combination on average is selected.

The test set is still left out, so that it gives an unbiased evaluation of the final model.

In [ ]:
param_grid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.0, 0.01, 0.1]) \
    .addGrid(lr.elasticNetParam, [0.0, 1.0]) \
    .build()

cv = CrossValidator(
    estimator=lr_pipeline,
    estimatorParamMaps=param_grid,
    evaluator=auc_evaluator,
    numFolds=3,
    seed=SEED,
)

cv_model = cv.fit(df_train)

for params, auc in zip(param_grid, cv_model.avgMetrics):
    readable = {param.name: value for param, value in params.items()}
    print(f"{readable} -> AUC: {auc:.3f}")

`cv_model.bestModel` is the pipeline retrained on the whole training set with the best hyperparameters. We evaluate it on the test set:

In [ ]:
best_model = cv_model.bestModel
best_predictions = best_model.transform(df_test)

print(f"Best model - AUC: {auc_evaluator.evaluate(best_predictions):.3f}")

## 9. Saving and loading the model

A `PipelineModel` can be saved to disk and loaded later (e.g. in another notebook or application) to make predictions on new data, without retraining it.

In [ ]:
model_path = str(OUTPUT_FILEPATH / "churn_model")

best_model.write().overwrite().save(model_path)

loaded_model = PipelineModel.load(model_path)

The loaded model includes all the preparation steps, so it can be applied directly to raw data. Here we score two new customers:

In [ ]:
new_customers = spark.createDataFrame(
    data=[
        ("NEW1", 25, "basic", 2, 15.0, 5),
        ("NEW2", 50, "premium", 48, 40.0, 0),
    ],
    schema="customer_id string, age int, plan string, tenure_months int, monthly_spend double, support_tickets int",
)

loaded_model.transform(new_customers) \
    .select("customer_id", "plan", "tenure_months", "support_tickets", "probability", "prediction") \
    .show(truncate=False)

## 10. All done!

**Next steps:**
- Add new features (e.g. `monthly_spend * tenure_months`, the total spent) with `withColumn` and see how the AUC changes
- Try other classifiers: `GBTClassifier`, `DecisionTreeClassifier`, `LinearSVC`
- Look at the jobs launched by `CrossValidator` in the Spark UI

When done working, close the SparkSession to release resources:

In [ ]:
spark.stop()